In [ ]:
import re
import time
import os
import sys
import pandas as pd
from dotenv import load_dotenv
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support.ui import Select
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# --- Configurações iniciais ---

dotenv_path = "../data/secure/.env"
load_dotenv(dotenv_path)
EMAIL = os.getenv("LOGIN2")
SENHA = os.getenv("SENHA_ISSE")

# Configurações do navegador Chrome
options = Options()
options.add_argument("--start-maximized")

# Detecta o sistema operacional e configura o driver apropriado
if sys.platform == "win32":
    # Windows: usa o caminho manual (mantém compatibilidade)
    chromedriver_path = r"C:\Users\User\Desktop\Repositorios\Automações\src\others\chromedriver.exe"
    service = Service(chromedriver_path)
    driver = webdriver.Chrome(service=service, options=options)
else:
    # Mac/Linux: usa webdriver-manager (baixa automaticamente)
    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=options)

# --- Login ---

driver.get("https://sso.maringa.pr.gov.br/auth/realms/maringa-externo/protocol/openid-connect/auth?response_type=code&client_id=nfse-maringa&redirect_uri=https://maringa.fintel.com.br/Account/OxyOpenId&state=Nfs")

WebDriverWait(driver, 15).until(EC.presence_of_element_located((By.XPATH, "//*[@id='username']"))).send_keys(EMAIL)
driver.find_element(By.XPATH, "//*[@id='password']").send_keys(SENHA)
driver.find_element(By.XPATH, "//*[@id='password']").send_keys(Keys.RETURN)
time.sleep(5)

# --- Lê os dados da planilha ---
df = pd.read_csv("../data/output/clientes_com_cpf.csv")

# Motivo no <select id="Motivo">: posição 1-based da <option> (//*[@id="Motivo"]/option[N]).
# Padrão 3 = option[3]; altere aqui quando precisar outro motivo.
MOTIVO_OPCAO_INDICE = 3

# Texto em //*[@id="DescricaoMotivo"] — altere quando precisar outro texto.
DESCRICAO_MOTIVO = (
    "Nota fiscal duplicada, serviços prestados em dezembro e lançados em duplicata por engano agora em março"
)

# DataFrames para guardar resultados
df_sem_cpf = pd.DataFrame(columns=df.columns)
df_notas_removidas = pd.DataFrame(columns=df.columns)

for index, row in df.iterrows():
    time.sleep(1.5)
    driver.get("https://maringa.fintel.com.br/ConsultaNotasFiscaisEmitidas")
    time.sleep(1.5)

    cpf = row["CPF"]
    if cpf == "000.000.000-00":
        print(f"CPF inválido, pulando: {cpf} (linha {index})")
        df_sem_cpf = pd.concat([df_sem_cpf, pd.DataFrame([row])], ignore_index=True)
        continue

    print(f"Processando exclusão — CPF: {cpf} (linha {index})")

    # Filtro por CPF/CNPJ na consulta (//*[@id="Filtro_Cnpj"])
    filtro_cnpj = WebDriverWait(driver, 15).until(
        EC.presence_of_element_located((By.ID, "Filtro_Cnpj"))
    )
    filtro_cnpj.clear()
    filtro_cnpj.send_keys(cpf)

    # Botão pesquisar / aplicar filtro — //*[@id="FormPrincipal"]/div/div[2]/div/button[1]
    btn_filtrar = WebDriverWait(driver, 15).until(
        EC.element_to_be_clickable(
            (By.XPATH, '//*[@id="FormPrincipal"]/div/div[2]/div/button[1]')
        )
    )
    driver.execute_script("arguments[0].scrollIntoView(true);", btn_filtrar)
    time.sleep(0.3)
    driver.execute_script("arguments[0].click();", btn_filtrar)
    time.sleep(1.5)

    # Link na primeira coluna da grid (não é <button>)
    # //*[@id="tabelaConsulta"]/table/tbody/tr/td[1]/div/a[1]
    link_nota = WebDriverWait(driver, 15).until(
        EC.element_to_be_clickable(
            (By.XPATH, '//*[@id="tabelaConsulta"]/table/tbody/tr/td[1]/div/a[1]')
        )
    )
    driver.execute_script("arguments[0].scrollIntoView(true);", link_nota)
    time.sleep(0.3)
    driver.execute_script("arguments[0].click();", link_nota)
    time.sleep(2)

    # //*[@id="acoes_usuario"]/button[2]
    btn_acao = WebDriverWait(driver, 15).until(
        EC.element_to_be_clickable(
            (By.XPATH, '//*[@id="acoes_usuario"]/button[2]')
        )
    )
    driver.execute_script("arguments[0].scrollIntoView(true);", btn_acao)
    time.sleep(0.3)
    driver.execute_script("arguments[0].click();", btn_acao)
    time.sleep(2)

    # Campo motivo — //*[@id="Motivo"]; opção //*[@id="Motivo"]/option[N] (N = MOTIVO_OPCAO_INDICE)
    select_motivo = WebDriverWait(driver, 15).until(
        EC.presence_of_element_located((By.ID, "Motivo"))
    )
    Select(select_motivo).select_by_index(MOTIVO_OPCAO_INDICE - 1)

    # //*[@id="DescricaoMotivo"]
    campo_desc_motivo = WebDriverWait(driver, 15).until(
        EC.presence_of_element_located((By.ID, "DescricaoMotivo"))
    )
    campo_desc_motivo.clear()
    campo_desc_motivo.send_keys(DESCRICAO_MOTIVO)

    # Confirmar envio — //*[@id="conteudo"]/div/div/div/div[2]/form/div/div[4]/button[2]
    btn_enviar = WebDriverWait(driver, 15).until(
        EC.element_to_be_clickable(
            (By.XPATH, '//*[@id="conteudo"]/div/div/div/div[2]/form/div/div[4]/button[2]')
        )
    )
    driver.execute_script("arguments[0].scrollIntoView(true);", btn_enviar)
    time.sleep(0.3)
    driver.execute_script("arguments[0].click();", btn_enviar)

    # Aguarda carregar; a próxima iteração volta ao início (ConsultaNotasFiscaisEmitidas)
    WebDriverWait(driver, 60).until(
        lambda d: d.execute_script("return document.readyState") == "complete"
    )
    time.sleep(2)

    df_notas_removidas = pd.concat(
        [df_notas_removidas, pd.DataFrame([row])], ignore_index=True
    )
    print(f"Exclusão enviada — CPF {cpf}. Próximo: recomeça na consulta.")

os.makedirs("../data/output", exist_ok=True)
df_sem_cpf.to_csv("../data/output/sem_cpf_remove.csv", index=False)
if not df_notas_removidas.empty:
    df_notas_removidas.to_csv("../data/output/notas_removidas.csv", index=False)
print("Encerrado.")
